<div style="display: flex; align-items: center; justify-content: flex-start; text-align: left;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Logo_universidad_icesi.svg/960px-Logo_universidad_icesi.svg.png" width="300" style="margin-right: 20px;">
    <div>
        <h3 style="margin: 0;">FACULTAD BARBERI DE INGENIERÍA, DISEÑO Y CIENCIAS APLICADAS</h3>
        <h3 style="margin: 0;">ALGORITMOS Y PROGRAMACIÓN III</h3>
    </div>
</div>

# Proyecto Final

Integrantes:
- ANGY MARIA HURTADO OSORIO A00401755
- HIDEKI TAMURA HERNANDEZ A00348618
- DAVID VERGARA LAVERDE A00402237


## Modelado - Experimentos de Deep Learning (CNN)

Este notebook se enfoca en la construcción de una Red Neuronal Convolucional (CNN) personalizada utilizando **TensorFlow/Keras**. El objetivo es crear un modelo de múltiples salidas (multi-output) que reciba la imagen original y prediga simultáneamente la **Calidad** y el **Tamaño**.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# TensorFlow y Keras para Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Añadir la carpeta raíz al path para importar src
sys.path.append(os.path.abspath('..'))
from src.evaluation.evaluate import evaluate_multioutput_model

### 1. Carga de Datos y Preprocesamiento de Imágenes

Para Deep Learning, en lugar de features extraídos manualmente, introducimos los píxeles de las imágenes redimensionadas (ej. 128x128x3). Aquí generaremos un conjunto de datos ficticio con tensores 4D para poder estructurar y probar la arquitectura.

In [ ]:
# TODO: Utilizar `tf.keras.preprocessing.image_dataset_from_directory` o src.utils.helpers para la carga real
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
NUM_SAMPLES = 500

np.random.seed(42)
# X será un tensor de forma (500, 128, 128, 3) simulando imágenes normalizadas entre 0 y 1
X_images = np.random.rand(NUM_SAMPLES, IMG_HEIGHT, IMG_WIDTH, CHANNELS).astype('float32')

# Etiquetas: 3 clases para Calidad, 3 clases para Tamaño
y_quality = np.random.choice([0, 1, 2], size=NUM_SAMPLES)
y_size = np.random.choice([0, 1, 2], size=NUM_SAMPLES)

# Partición de datos (entrenamiento y prueba)
X_train, X_test, y_train_qual, y_test_qual, y_train_size, y_test_size = train_test_split(
    X_images, y_quality, y_size, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train_qual shape: {y_train_qual.shape}, y_train_size shape: {y_train_size.shape}")

### 2. Definición de la Arquitectura CNN Bifurcada (Multi-Output)

Diseñamos una red con un **tronco común** (Common Feature Extractor) compuesto por capas convolucionales para aprender patrones visuales generales (bordes, colores, texturas). Luego, la red se divide en **dos ramas (cabezas)**: una para predecir la calidad y otra para el tamaño.

In [ ]:
def build_multioutput_cnn(input_shape=(128, 128, 3), num_quality_classes=3, num_size_classes=3):
    # --- Tronco Común (Feature Extractor) ---
    inputs = Input(shape=input_shape, name='imagen_entrada')
    
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    
    # Usamos GlobalAveragePooling en lugar de Flatten para reducir parámetros y mitigar overfitting
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x) # Regularización
    
    # --- Rama 1: Predicción de Calidad ---
    quality_branch = Dense(64, activation='relu')(x)
    quality_branch = Dropout(0.3)(quality_branch)
    quality_output = Dense(num_quality_classes, activation='softmax', name='quality_output')(quality_branch)
    
    # --- Rama 2: Predicción de Tamaño ---
    size_branch = Dense(64, activation='relu')(x)
    size_branch = Dropout(0.3)(size_branch)
    size_output = Dense(num_size_classes, activation='softmax', name='size_output')(size_branch)
    
    # --- Ensamblaje del Modelo ---
    model = Model(inputs=inputs, outputs=[quality_output, size_output], name='MultiOutput_CNN')
    return model

cnn_model = build_multioutput_cnn()
cnn_model.summary()

### 3. Compilación del Modelo

Configuramos la función de pérdida (`loss`) para ambas salidas y asignamos pesos a cada pérdida si fuera necesario. En este caso daremos el mismo peso (1.0) a ambas tareas.

In [ ]:
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        'quality_output': 'sparse_categorical_crossentropy',
        'size_output': 'sparse_categorical_crossentropy'
    },
    loss_weights={
        'quality_output': 1.0,
        'size_output': 1.0
    },
    metrics={
        'quality_output': ['accuracy'],
        'size_output': ['accuracy']
    }
)

### 4. Entrenamiento con Callbacks (Previniendo Overfitting)

Añadimos `EarlyStopping` para detener el entrenamiento si la métrica en validación no mejora, y `ReduceLROnPlateau` para reducir la tasa de aprendizaje si el modelo se estanca.

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss')
]

history = cnn_model.fit(
    X_train,
    {'quality_output': y_train_qual, 'size_output': y_train_size},
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

### 5. Curvas de Aprendizaje

Es crucial graficar la pérdida global (loss) para detectar visualmente el sobreajuste.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Pérdida de Entrenamiento (Total)')
plt.plot(history.history['val_loss'], label='Pérdida de Validación (Total)')
plt.title('Curvas de Pérdida del Modelo CNN Multi-Salida')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.show()

### 6. Evaluación del Modelo

Realizamos las predicciones sobre el conjunto de test. Keras nos devolverá una lista con dos arrays de probabilidades (uno para calidad y otro para tamaño). Tomaremos el `argmax` para obtener la clase y lo pasaremos a nuestra función centralizada.

In [ ]:
# Realizar predicciones sobre los datos de prueba
predictions = cnn_model.predict(X_test)
pred_probs_quality = predictions[0]
pred_probs_size = predictions[1]

# Convertir probabilidades a índices de clases usando argmax
y_pred_quality = np.argmax(pred_probs_quality, axis=1)
y_pred_size = np.argmax(pred_probs_size, axis=1)

quality_names = ['Mala', 'Regular', 'Buena']
size_names = ['Pequeño', 'Mediano', 'Grande']

# Evaluación usando src/evaluation/evaluate.py
evaluate_multioutput_model(
    y_test_qual, y_pred_quality,
    y_test_size, y_pred_size,
    quality_classes=quality_names,
    size_classes=size_names
)

### 7. Guardar el Modelo Entrenado

Guardamos el modelo en formato `.h5` o `SavedModel` de Keras para su despliegue en la aplicación web.

In [ ]:
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'cnn_multioutput.keras') # Se recomienda .keras en nuevas versiones

cnn_model.save(model_path)
print(f"Modelo de Deep Learning guardado exitosamente en: {model_path}")